# 02 — LLM-Based Signal Extraction
**Demand Signal Feature Store — Capstone Project**

This notebook uses Azure OpenAI to extract structured signals from supplier notes:
- `risk_mentioned` (bool)
- `delay_days` (int)
- `capacity_flag` (normal / constrained / critical)
- `sentiment_score` (-1.0 to 1.0)
- `key_phrases` (list of strings)

Each extraction carries **lineage metadata**: timestamp, model version, source.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from src.llm_extractor import get_client, extract_batch
from config.settings import EXTRACTION_SCHEMA

## 1. Load Supplier Notes

In [ ]:
notes_df = pd.read_csv("../data/supplier_notes.csv", parse_dates=["note_date"])
print(f"Loaded {len(notes_df)} supplier notes")
notes_df.head()

## 2. Review Extraction Schema
This is the fixed JSON schema that constrains the LLM output.

In [ ]:
import json
print(json.dumps(EXTRACTION_SCHEMA, indent=2))

## 3. Run Extraction

**Prerequisites:** Set your Azure OpenAI credentials in `.env` (see `.env.example`).

For cost management, you can extract a subset first to verify quality.

In [ ]:
# Test with a small sample first
sample = notes_df.head(10)
client = get_client()

print("Extracting from 10-note sample...")
sample_extracted = extract_batch(client, sample, batch_delay=0.3)
sample_extracted.head()

In [ ]:
# Verify extraction quality — spot-check a few
for _, row in sample.head(3).iterrows():
    extracted = sample_extracted[sample_extracted["note_id"] == row["note_id"]].iloc[0]
    print(f"\nNote: {row['note_text'][:100]}...")
    print(f"  risk_mentioned: {extracted['risk_mentioned']}")
    print(f"  delay_days: {extracted['delay_days']}")
    print(f"  capacity_flag: {extracted['capacity_flag']}")
    print(f"  sentiment_score: {extracted['sentiment_score']}")
    print(f"  key_phrases: {extracted['key_phrases']}")

In [ ]:
# Full extraction — uncomment when ready (takes ~5-10 min depending on note count)
# print(f"Running full extraction on {len(notes_df)} notes...")
# extracted_df = extract_batch(client, notes_df, batch_delay=0.3)
# print(f"Extraction complete: {len(extracted_df)} results")

## 4. Extraction Quality Summary

In [ ]:
# Use sample_extracted for now; replace with extracted_df after full run
df = sample_extracted.copy()

print("=== Extraction Summary ===")
print(f"Total notes processed: {len(df)}")
print(f"Extraction errors: {df['extraction_error'].notna().sum() if 'extraction_error' in df.columns else 0}")
print(f"Risk mentioned: {df['risk_mentioned'].sum()} ({df['risk_mentioned'].mean():.1%})")
print(f"Avg delay signal: {df['delay_days'].mean():.1f} days")
print(f"Avg sentiment: {df['sentiment_score'].mean():.2f}")
print(f"\nCapacity flag distribution:")
print(df["capacity_flag"].value_counts())

## 5. Save Extracted Features

In [ ]:
# Save — replace sample_extracted with extracted_df after full run
df.to_csv("../data/extracted_signals.csv", index=False)
print("Saved to ../data/extracted_signals.csv")

# Lineage check
print(f"\nLineage metadata:")
print(f"  Model: {df['model_version'].iloc[0]}")
print(f"  Source: {df['source'].iloc[0]}")
print(f"  Earliest extraction: {df['extraction_timestamp'].min()}")
print(f"  Latest extraction: {df['extraction_timestamp'].max()}")